In [1]:
# Ensure that we are using the correct host
import socket
try:
    assert "gpu" in socket.gethostname()
    print(f"Running on {socket.gethostname()}. All is good!")
except:
    raise RuntimeError(f"Be sure to run on GPU! You are currently running on {socket.gethostname()}")

Running on gpu41.storrs.hpc.uconn.edu. All is good!


In [2]:
import pandas as pd
import numpy as np
import pymc as pm
import arviz as az
from scipy import stats
import pytensor.tensor as pt

In [3]:
base = "/shared/healthinfolab/datasets/ABCD/Irritability/Clinical_Data/Irritability/Release_5.0/"

files = {
    0:  base + "abcd_cbcl_irr_index_release5.0_0m_long.csv",
    12: base + "abcd_cbcl_irr_index_release5.0_12m_long.csv",
    24: base + "abcd_cbcl_irr_index_release5.0_24m_long.csv",
    36: base + "abcd_cbcl_irr_index_release5.0_36m_long.csv",
    48: base + "abcd_cbcl_irr_index_release5.0_48m_long.csv",
}

In [4]:
dfs = []

for t, path in files.items():
    df = pd.read_csv(path)
    df = df[[
        "src_subject_id",
        "cbcl_irr_index_cnst"
    ]].copy()

    df["time"] = t

    dfs.append(df)

data = pd.concat(dfs, ignore_index=True)

In [5]:
data = data.rename(columns={
    "src_subject_id": "id",
    "cbcl_irr_index_cnst": "irr"
})

data = data.dropna()

counts = data.groupby("id").size()
valid_ids = counts[counts >= 2].index
data = data[data["id"].isin(valid_ids)]

In [6]:
# choose a small subset of subjects
subset_size = 1000

np.random.seed(42)  # set seed for reproducibility

subset_ids = np.random.choice(
    data["id"].unique(),
    size=subset_size,
    replace=False
)

data = data[data["id"].isin(subset_ids)].copy()


In [7]:
ids = data["id"].unique()
id_map = {v:i for i,v in enumerate(ids)}

data["id_i"] = data["id"].map(id_map)

# Standardize time to [0, 1] so slope priors are on a sensible scale
data["time_std"] = data["time"] / 48.0

time = data["time_std"].values
y = data["irr"].values
person = data["id_i"].values

N = len(ids)


In [ ]:
K = 4

with pm.Model() as model:
    pi        = pm.Dirichlet("pi", np.ones(K))
    intercept_raw = pm.Normal("intercept_raw", 0, 3, shape=K)
    intercept     = pm.Deterministic("intercept", pt.sort(intercept_raw))
    slope     = pm.Normal("slope", 0, 1, shape=K)
    sigma     = pm.HalfNormal("sigma", 1, shape=K)

    # Compute log-likelihood for each observation under each class
    # Shape: (n_obs, K)
    def log_likelihood_k(k):
        mu_k = intercept[k] + slope[k] * time
        return pm.logp(pm.Normal.dist(mu_k, sigma[k]), y)

    # Stack into (n_obs, K) and sum over each person's observations per class
    # We need per-person log-likelihoods, so we use person indices to sum
    loglik_obs = pt.stack([log_likelihood_k(k) for k in range(K)], axis=1)  # (n_obs, K)

    # Pre-build a (N, n_obs) indicator matrix in numpy — person i observed at which rows
    person_matrix = (np.arange(N)[:, None] == person[None, :]).astype(float)  # (N, n_obs)

    # Sum each person's per-observation log-likelihoods across classes: (N, K)
    loglik_person = pt.dot(person_matrix, loglik_obs)  # (N, K)

    # Log mixture: log sum_k pi_k * p(y_i | k)
    log_mix = pt.logsumexp(pt.log(pi) + loglik_person, axis=1)
    pm.Potential("obs", log_mix.sum())

    trace = pm.sample(
        1000,
        tune=1000,
        target_accept=0.95,
        chains=4,
        random_seed=42
    )

AttributeError: 'bool' object has no attribute 'T'

In [ ]:
# Convergence diagnostics
summary = az.summary(trace, var_names=["intercept", "slope", "sigma", "pi"])
print(summary)

# Flag any parameters with R-hat > 1.01 (indicates poor convergence)
bad_rhat = summary[summary["r_hat"] > 1.01]
if len(bad_rhat) > 0:
    print("\nWARNING: The following parameters have R-hat > 1.01 (poor convergence):")
    print(bad_rhat[["r_hat"]])
else:
    print("\nAll R-hat values <= 1.01. Convergence looks good.")


In [ ]:
import matplotlib.pyplot as plt

posterior = trace.posterior

inter = posterior["intercept"].mean(("chain", "draw")).values
slope = posterior["slope"].mean(("chain", "draw")).values

# time is standardized to [0,1]; plot in original months for interpretability
t_std = np.linspace(0, 1, 100)
t_months = t_std * 48

for k in range(K):
    plt.plot(t_months, inter[k] + slope[k] * t_std, label=f"Class {k+1}")

plt.xlabel("Months")
plt.ylabel("Irritability")
plt.title("LCGA Trajectories by Class")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
# Compute posterior class probabilities for each person
intercept_post = trace.posterior["intercept"].mean(("chain", "draw")).values  # (K,)
slope_post     = trace.posterior["slope"].mean(("chain", "draw")).values
sigma_post     = trace.posterior["sigma"].mean(("chain", "draw")).values
pi_post        = trace.posterior["pi"].mean(("chain", "draw")).values

log_probs = np.zeros((N, K))
for k in range(K):
    for i in range(N):
        mask = person == i
        mu_k = intercept_post[k] + slope_post[k] * time[mask]
        log_probs[i, k] = np.sum(stats.norm.logpdf(y[mask], mu_k, sigma_post[k]))

log_probs += np.log(pi_post)
classes = np.argmax(log_probs, axis=1) + 1  # 1-indexed

class_df = pd.DataFrame({"id": ids, "class": classes})
print(class_df["class"].value_counts().sort_index())
class_df